In [1]:
using LinearAlgebra

# Matrix multiplication benchmark
println("Testing matrix multiplication speed...")
A = rand(1000, 1000)
B = rand(1000, 1000)
@time A * B

# Expected times:
# Modern desktop: 10-30 ms
# Decent laptop: 30-100 ms
# Older laptop: 100-300 ms

Testing matrix multiplication speed...
  1.018430 seconds (4.48 M allocations: 230.448 MiB, 7.12% gc time, 93.27% compilation time)


1000×1000 Matrix{Float64}:
 244.478  240.124  247.675  241.92   …  238.272  239.116  247.706  244.935
 246.124  244.127  244.464  253.265     241.426  240.409  254.43   250.667
 262.854  253.431  250.918  253.236     251.949  243.021  256.787  250.58
 248.682  239.827  246.76   246.546     242.556  237.903  250.034  243.446
 242.569  227.011  236.684  237.408     235.3    230.725  244.058  232.904
 253.242  242.617  252.244  251.421  …  249.819  241.603  254.286  247.984
 255.261  248.411  252.143  245.502     247.43   244.686  258.606  249.728
 260.061  248.194  251.119  248.624     247.92   246.723  255.675  246.571
 251.5    240.241  245.759  245.994     241.899  243.981  250.168  246.094
 256.458  251.357  249.628  252.668     254.997  254.517  261.534  253.532
 256.833  247.108  245.032  250.939  …  247.512  247.154  256.935  243.205
 263.753  250.553  255.309  255.093     252.601  252.187  259.729  258.474
 253.943  250.022  252.333  253.369     246.73   241.745  252.131  255.287

In [2]:
# Simple ODE benchmark
using DifferentialEquations

function lorenz!(du, u, p, t)
    du[1] = 10.0 * (u[2] - u[1])
    du[2] = u[1] * (28.0 - u[3]) - u[2]
    du[3] = u[1] * u[2] - (8/3) * u[3]
end

u0 = [1.0, 0.0, 0.0]
tspan = (0.0, 100.0)
prob_bench = ODEProblem(lorenz!, u0, tspan)

println("\nTesting ODE solver speed (Lorenz system)...")
@time sol_bench = solve(prob_bench, Rodas5())

# Expected times:
# Modern desktop: 0.01-0.05 seconds
# Decent laptop: 0.05-0.2 seconds
# Older laptop: 0.2-1.0 seconds


Testing ODE solver speed (Lorenz system)...
  3.373295 seconds (4.43 M allocations: 233.726 MiB, 3.08% gc time, 99.59% compilation time)


retcode: Success
Interpolation: specialized 4rd order "free" stiffness-aware interpolation
t: 1422-element Vector{Float64}:
   0.0
   3.5678604836301404e-5
   0.0003924646531993154
   0.003219523954374159
   0.010130792498516391
   0.019221699189546783
   0.03131202799578121
   0.04678203688141252
   0.06638539030431816
   0.09013871628784907
   0.11752372077027054
   0.14771260886765322
   0.18025028950642913
   ⋮
  99.31722678454234
  99.36622716669002
  99.4152275488377
  99.46876301934203
  99.52229848984636
  99.58672056846802
  99.6610901903651
  99.73545981226218
  99.80982943415925
  99.88419905605633
  99.96284280554244
 100.0
u: 1422-element Vector{Vector{Float64}}:
 [1.0, 0.0, 0.0]
 [0.9996434557625105, 0.0009988049817849076, 1.7814347887992186e-8]
 [0.996104549742581, 0.010965399721242447, 2.146955365852547e-6]
 [0.9697366941267918, 0.0886102562861196, 0.00014010971935064003]
 [0.9170749617292998, 0.269664830773501, 0.0012956125778399218]
 [0.8707747383226055, 0.49318966291

In [3]:
# Check Julia threads and system info
println("\nSystem information:")
println("Julia version: ", VERSION)
println("Number of threads: ", Threads.nthreads())
println("CPU: ", Sys.cpu_info()[1].model)
println("Total RAM: ", round(Sys.total_memory() / 2^30, digits=1), " GB")

# Check if you're using optimized BLAS
using LinearAlgebra
println("BLAS config: ", BLAS.get_config())


System information:
Julia version: 1.11.6
Number of threads: 1
CPU: Intel(R) Core(TM) i7-9750H CPU @ 2.60GHz
Total RAM: 15.8 GB
BLAS config: LBTConfig([ILP64] libopenblas64_.dll)


In [4]:
# Simplified reaction network benchmark
function simple_reactions!(du, u, p, t)
    k = [1.0, 0.1, 0.5]  # Rate constants
    
    # A + B -> C (k1)
    rate1 = k[1] * u[1] * u[2]
    # C -> A + B (k2)
    rate2 = k[2] * u[3]
    # C -> D (k3)
    rate3 = k[3] * u[3]
    
    du[1] = -rate1 + rate2  # A
    du[2] = -rate1 + rate2  # B
    du[3] = rate1 - rate2 - rate3  # C
    du[4] = rate3  # D
end

# 10 nodes, 4 species each = 40 ODEs
u0_bench = repeat([1.0, 1.0, 0.0, 0.0], 10)
prob_chem = ODEProblem(simple_reactions!, u0_bench, (0.0, 10.0))

println("\nTesting 40-ODE chemical system...")
@time sol_chem = solve(prob_chem, Rodas5(), abstol=1e-6, reltol=1e-4)

# Expected times:
# Modern desktop: 0.05-0.2 seconds
# Decent laptop: 0.2-0.5 seconds
# Older laptop: 0.5-2.0 seconds


Testing 40-ODE chemical system...
  2.256337 seconds (858.51 k allocations: 49.602 MiB, 20.68% compilation time)


retcode: Success
Interpolation: specialized 4rd order "free" stiffness-aware interpolation
t: 19-element Vector{Float64}:
  0.0
  0.029898066669076923
  0.08971480049677882
  0.17441658197124613
  0.2869952100814324
  0.4347034678033449
  0.6232228328065331
  0.8617031643969937
  1.1607319805603071
  1.5356666008081208
  2.0073245845821193
  2.6035912780436763
  3.355244084807384
  4.2849015891270525
  5.349575949795799
  6.524855670201755
  7.8164327508349
  9.2252342669087
 10.0
u: 19-element Vector{Vector{Float64}}:
 [1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0  …  0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0]
 [0.9710125974248383, 0.9710125974248383, 0.028769593299751928, 0.00021780927540978999, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0  …  0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0]
 [0.9180239751571985, 0.9180239751571985, 0.08011049913747258, 0.0018655257053288392, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0  …  0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0]
 [0.8526740746476047, 0.852674